# Quick Tutorial

This notebook aims to show how to generate Life-Cycle Assessment (LCA) impact scores to be used in an Energy System Model (ESM). LCA impact scores can be used within modelling constraints, e.g., upper limit on life-cycle greenhouse gas emissions, or in the objective function, e.g., minimizing the total damage on human health. _mescal_ applies several transformations on LCA data to ensure the alignement between your ESM and LCA models. For instance, _mescal_ performs double-counting removal (to avoid the overestimation of flows that are already represented in your ESM, e.g., energy flows), technological parameters harmonization (e.g., lifetime, efficiency, capacity factors), and normalization (to ease the solving process in your ESM).

In this notebook, we illustrate the use of _mescal_ with the core model of [REHO](https://reho.readthedocs.io/en/main/). We employed the ecoinvent LCA database, but the overall methodology is agnostic to the used LCA database. To replicate this example for your ESM, you should adapt the input data files accordingly, following the example ones.

We show how to:
- create the ESM database (i.e., the database containing the datasets corresponding to the technologies and resources of the model) in your brightway2 project
- perform life-cycle impact assessment and contribution analyses
- create the .mod and .dat files for your ESM
- visualise the results using _mescal_'s [embodied plots](https://github.com/matthieu-str/mescal/blob/master/dev/plot.ipynb)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%pip install mescal==1.2.3

Note: you may need to restart the kernel to use updated packages.


## Project setup

In [3]:
# Import the required libraries
from mescal import *
import pandas as pd
import bw2data as bd

In [4]:
ecoinvent_version = '3.10.1' # choose the ecoinvent version you wish to use
esm_location = 'CH' # choose the version of energyscope for which you want to generate metrics
# spatialized_database = True # set to True if you want to use your spatialized version of ecoinvent
spatialized_database = False # set to True if you want to use your spatialized version of ecoinvent
#regionalize_foregrounds = ['Operation', 'Resource'] #['Operation'] # set to 'all' if you want to regionalize the foreground inventories of all types of LCI datasets
premise_iam = 'image' # choose the IAM to which the premise database is linked
premise_ssp_rcp = 'SSP2-L' # choose the SSP/RCP scenario to which the premise database is linked
year = 2050 # choose the year for which you want to generate the new database with the ESM results

In [5]:
# Set the name of your main LCI database (e.g., ecoinvent or premise database) here:
lca_db_name = "ecoinvent-3.10.1-cutoff"

lca_db_name = f"ecoinvent_cutoff_{ecoinvent_version}_{premise_iam}_{premise_ssp_rcp}_{year}"
print(f'Your main LCI database is: {lca_db_name}')

Your main LCI database is: ecoinvent_cutoff_3.10.1_image_SSP2-L_2050


In [6]:
# Set the name of the new database with the ESM results
esm_db_name = f'REHO_{esm_location}_{year}'

In [7]:
# Set the list of LCIA methods for which you want indicators (they must be registered in your brightway project)
lcia_methods=['EF v3.1']

In [8]:
# Set up your Brightway project
bd.projects.set_current('ecoinvent3.10.1') # put the name of your brightway project here

In [9]:
# Load the LCI database from your brightway project
lca_db = Database(lca_db_name, create_pickle=True)

2026-03-17 14:32:03,518 - Database - INFO - Loaded ecoinvent_cutoff_3.10.1_image_SSP2-L_2050 from pickle!


## Input data

In [ ]:
path_to_input_files = 'data/lca/'  # put here the path to the folder containing the input data files

Input data files can be retrieved from [mescal GitHub repository](https://github.com/matthieu-str/mescal/tree/master/dev/REHO_data/core)

In [11]:
mapping = pd.read_csv(path_to_input_files+'mapping.csv')
unit_conversion = pd.read_excel(path_to_input_files+'unit_conversion.xlsx')
mapping_esm_flows_to_CPC = pd.read_csv(path_to_input_files+'mapping_esm_flows_to_CPC.csv')
model = pd.read_csv(path_to_input_files+'model.csv')
technology_compositions = pd.read_csv(path_to_input_files+'technology_compositions.csv')
technology_specifics = pd.read_csv(path_to_input_files+'technology_specifics.csv')
lifetime = pd.read_csv(path_to_input_files+'lifetime.csv')
efficiency = pd.read_csv(path_to_input_files+'efficiency.csv') 
impact_abbrev = pd.read_csv(path_to_input_files+'impact_abbrev.csv')

In [12]:
mapping.Database = lca_db_name  # set the database name in the mapping file

In [13]:
# Define the user-defined ranking
if esm_location == 'CA-QC':
    my_ranking = [
        'CA-QC', # Quebec
        'CA', # Canada
        'CAN', # Canada in IMAGE
        'CAZ', # Canada - Australia - New Zealand in REMIND
        'RNA', # North America
        'US', # United States
        'USA', # United States in REMIND and IMAGE
        'GLO', # Global average
        'RoW', # Rest of the world
    ]
elif esm_location == 'CH':
    my_ranking = [
        'CH',
        'NEU',
        'EUR',
        'WEU',
        'RER',
        'IAI Area, EU27 & EFTA',
        'GLO',
        'RoW'
    ]
if esm_location == "EU":
    my_ranking = [
        "RER",
        "EUR",                                    
        "Europe without Switzerland",
        "Europe without Austria",
        "Europe without NORDEL",
        "WEU",                  # Western Europe
        "CEU",                  # Central Europe
        "NEU",                  # Northern Europe
        "SEU",                  # Southern Europe
        "IAI Area, EU27 & EFTA",
        "GLO",                  # global average
        "RoW"                   # rest of world
    ]
else:
    my_ranking = [
        'GLO',
        'RoW',
    ]

## Initialize the ESM class

In [14]:
technology_specifics

,Name,Specifics,Amount,Comment,Comment
0,ATM_CCS,DAC,NaN,NaN,NaN
1,BIOMETHANATION,Process,NaN,NaN,NaN
2,BIO_HYDROLYSIS,Process,NaN,NaN,NaN
3,DEC_DIRECT_ELEC,No background search,NaN,NaN,NaN
4,ELECTRICITY,Import/Export,NaN,NaN,NaN
5,ELEC_EXPORT,Import/Export,NaN,NaN,NaN
6,GASIFICATION_SNG,Background search,3.0,NaN,NaN
7,H2_ELECTROLYSIS,Decommissioning,NaN,NaN,NaN
8,HABER_BOSCH,Process,NaN,NaN,NaN
9,IND_DIRECT_ELEC,No background search,NaN,NaN,NaN


In [ ]:
esm = ESM(
    mapping=mapping,
    unit_conversion=unit_conversion,
    model=model,
    mapping_esm_flows_to_CPC_cat=mapping_esm_flows_to_CPC,
    main_database=lca_db,
    esm_db_name=esm_db_name,
    esm_location=esm_location,
    technology_compositions=technology_compositions,
    tech_specifics=technology_specifics,
    lifetime=lifetime,
    efficiency=efficiency,
    regionalize_foregrounds=['Operation', 'Resource'],  # types of LCI datasets that will be regionalized
    results_path_file='data/lca/',
    locations_ranking=my_ranking,

)

In [16]:
esm.clean_inputs()

In [17]:
# Update mapping dataframe with better locations
esm.change_location_mapping_file()

2026-03-17 14:32:04,723 - Mescal - WARNING - No location found in your ranking for (transport, passenger car, transport, passenger car, battery electric, Medium) in the database ecoinvent_cutoff_3.10.1_image_SSP2-L_2050. Have to keep the initial location: KOR
2026-03-17 14:32:04,960 - Mescal - WARNING - No location found in your ranking for (electricity, low voltage, market for electricity, low voltage) in the database ecoinvent_cutoff_3.10.1_image_SSP2-L_2050. Have to keep the initial location: CA-NS
2026-03-17 14:32:05,239 - Mescal - WARNING - No location found in your ranking for (electricity, medium voltage, market for electricity, medium voltage) in the database ecoinvent_cutoff_3.10.1_image_SSP2-L_2050. Have to keep the initial location: SA
2026-03-17 14:32:05,404 - Mescal - WARNING - No location found in your ranking for (electricity, low voltage, electricity production, photovoltaic, residential) in the database ecoinvent_cutoff_3.10.1_image_SSP2-L_2050. Have to keep the initia

In [18]:
esm.main_database.test_mapping_file(esm.mapping)  # test the mapping file

2026-03-17 14:32:05,673 - Database - INFO - Mapping successfully linked to the database


[]

In [19]:
esm.check_inputs()

2026-03-17 14:32:05,730 - Mescal - WARNING - List of technologies or resources that are in the model file but not in the mapping file. Their impact scores will be set to the default value: ['DHN_hex']
2026-03-17 14:32:05,745 - Mescal - WARNING - Some technologies have no lifetime value for LCA in the lifetime file. Therefore, lifetime harmonization with the ESM will not be performed during the LCIA phase and capacity factor harmonization during the feedback of ESM results will not be performed either for those technologies: ['ElectricalHeater_SH', 'ElectricalHeater_other', 'ElectricalHeater_DHW', 'ThermalSolar', 'HeatPump_Geothermal_district', 'WaterTankSH', 'WaterTankDHW', 'ElectricalHeater_other_district']
2026-03-17 14:32:05,763 - Mescal - WARNING - List of technologies that are in the tech_specifics file but not in the mapping file (this will not be a problem in the workflow): ['ATM_CCS', 'BIOMETHANATION', 'BIO_HYDROLYSIS', 'DEC_DIRECT_ELEC', 'ELECTRICITY', 'ELEC_EXPORT', 'GASIFICA

## Create the ESM database in your Brightway2 project

In [20]:
esm.create_esm_database()

2026-03-17 14:32:08,693 - Mescal - INFO - Starting to remove double-counted flows
2026-03-17 14:32:09,556 - Mescal - WARNING - No location found in your ranking for (transport, freight, lorry, market for transport, freight, lorry) in the database ecoinvent_cutoff_3.10.1_image_SSP2-L_2050. Have to keep the initial location: World
100%|██████████| 18/18 [00:03<00:00,  5.69it/s]
2026-03-17 14:32:14,908 - Mescal - INFO - Double-counting removal done in 6.2 seconds
2026-03-17 14:32:14,920 - Mescal - INFO - Starting to correct efficiency differences
2026-03-17 14:32:15,169 - Mescal - WARNING - No flow found for type(s) ['Wood'] in WOOD_Stove. The efficiency of this technology cannot be adjusted.
2026-03-17 14:32:15,575 - Mescal - WARNING - No flow found for type(s) ['Electricity'] in EV_district. The efficiency of this technology cannot be adjusted.
2026-03-17 14:32:15,623 - Mescal - WARNING - No flow found for type(s) ['Gasoline'] in ICE_district. The efficiency of this technology cannot be

Title: Writing activities to SQLite3 database:
  Started: 03/17/2026 14:32:17
  Finished: 03/17/2026 14:32:17
  Total time elapsed: 00:00:00
  CPU %: 93.50
  Memory %: 9.71


2026-03-17 14:32:17,978 - Database - INFO - REHO_CH_2050 written to Brightway!
2026-03-17 14:32:17,981 - Mescal - INFO - Database written in 0.9 seconds


## Compute LCA impact scores and perform contribution analysis

In [21]:
impact_scores, contrib_analysis_res, _ = esm.compute_impact_scores(
    methods=lcia_methods,
    impact_abbrev=impact_abbrev,
    contribution_analysis='both',
)   # >5min

Getting activity data


100%|██████████| 52/52 [00:00<00:00, 28712.98it/s]


Adding exchange data to activities


100%|██████████| 1096/1096 [00:00<00:00, 65319.96it/s]


Filling out exchange data


100%|██████████| 52/52 [00:00<00:00, 300.96it/s]
2026-03-17 14:32:18,344 - Database - INFO - Loaded REHO_CH_2050 from brightway!
52it [00:14,  3.60it/s]


In [22]:
specific_lcia_abbrev =  ['Aci', 'CC', 'ETF', 'ERNR',  'EUTf',  'EUTm',  'EUTt',  'HTC', 'HTNC', 'IR',  'LU',  'MR',  'OD',  'PMF',  'POF',  'WU']

In [23]:
df_impact = esm.normalize_lca_metrics(
    R=impact_scores,
    mip_gap=1e-3,   # the threshold will be reapplied while loading in REHO
    lcia_methods=lcia_methods,
    specific_lcia_abbrev=specific_lcia_abbrev,
    impact_abbrev=impact_abbrev,
    file_name='techs_lca',
    skip_normalization=True,
    output='return',
)

df_impact.to_csv(esm.results_path_file+'LCA_impacts.csv', index=False)

## Visualize the LCA impact scores

In [24]:
plot = Plot(
    df_impact_scores=impact_scores,
    lifetime=lifetime,  # used to visualise infrastructure impacts per kW.year
)

In [25]:
visualise_technologies = ['NG_Boiler'
,'OIL_Boiler'
,'WOOD_Stove'
,'HeatPump_Air'
# ,'HeatPump_Geothermal'
# ,'HeatPump_DHN'
# ,'HeatPump_Lake'
,'AirConditioner'
,'ElectricalHeater_other'
# ,'ElectricalHeater_SH'
# ,'ElectricalHeater_DHW'
# ,'EV_district'
# ,'EV_charger_district'
# ,'ICE_district'
,'NG_Boiler_district'
# ,'NG_Cogeneration_district'
# ,'HeatPump_Geothermal_district'
# ,'ElectricalHeater_other_district'
]

In [26]:
plot.plot_indicators_of_technologies_for_one_impact_category(
    technologies_list=visualise_technologies,
    impact_category=(
        'EF v3.1',
        'climate change',
        'global warming potential (GWP100)',
    ),
    metadata={
        'operation_unit': 'kWh',
        'construction_unit': 'kW',
        'technologies_type': 'electricity production',
    },
)

In [27]:
plot.plot_indicators_of_technologies_for_one_impact_category(
    technologies_list=visualise_technologies,
    impact_category=(
        'EF v3.1',
        'material resources: metals/minerals',
        'abiotic depletion potential (ADP): elements (ultimate reserves)',
    ),
    metadata={
        'operation_unit': 'kWh',
        'construction_unit': 'kW',
        'technologies_type': 'electricity production',
    },
)

In [28]:
plot.plot_indicators_of_technologies_for_several_impact_categories(
    technologies_list=visualise_technologies,
    impact_categories_list=[
        ('EF v3.1', 'particulate matter formation', 'impact on human health'),
        ('EF v3.1', 'climate change', 'global warming potential (GWP100)'),
        ('EF v3.1', 'material resources: metals/minerals', 'abiotic depletion potential (ADP): elements (ultimate reserves)'),
    ]
)

In [29]:
plot.plot_indicators_of_resources_for_several_impact_categories(
    resources_list=['NaturalGas', 'Electricity', 'Oil'],
    impact_categories_list=[
        ('EF v3.1', 'particulate matter formation', 'impact on human health'),
        ('EF v3.1', 'climate change', 'global warming potential (GWP100)'),
        ('EF v3.1', 'material resources: metals/minerals', 'abiotic depletion potential (ADP): elements (ultimate reserves)'),
    ]
)